# 04 — Robustness and open issues  [LIMITATIONS / APPENDIX]

The `rate_total` sensitivity check, the German/non-German directionality test and its US-classifier contamination, the classifier's own validation status, and the spatial dependence left in every model. Consolidated from `archive/supervisor_run_2026-08-16.ipynb`, `archive/rate_total_analysis.ipynb`, and `archive/ethnicity_validation_scoring.ipynb`.

## Contents

- **4.1** `rate_total` sensitivity
- **4.2** Reviewer-group moderation (renamed from "directionality")
- **4.3** The "United States" classifier bug
- **4.4** Classifier validation status — OPEN
- **4.5** Directionality: not yet tested
- **4.6** Spatial dependence

### 4.1 `rate_total` sensitivity  [LIMITATIONS — SETTLED]

Full 5-predictor `rate_total` fit vs. `rate_social`: same sign and overlapping CI on every one of the 5 predictors (§1.9's outcome decision rests on this). **AIC/dispersion note:** `rate_total`'s NB dispersion is 0.961 (well-specified); **AIC is not comparable across the two response variables** — only the two `rate_total` models (pruned-selection vs. full-5-predictor) can be compared to each other on AIC (2300.8 vs. 2287.8, §2.6). `share_social`'s naive-binomial dispersion (42.5) is very high; HC1 robust SEs mitigate this but the binomial variance assumption is clearly wrong there — treat its p-value, not its raw SEs, with care. *(Source: `rate_total_analysis.ipynb` C8, moved to `archive/`.)*

### 4.2 Reviewer-group moderation  [LIMITATIONS — SETTLED, one open thread]

*(Renamed from "directionality" — see §4.5: this test does not establish direction.)*

**4.2.1 Group construction, feasibility gate.** German vs. non-German (Arab+Türkiye+Other) vs. Unknown, by `ethnicity_final`. Gate: **PASS** — median `n_rac_nongerman`=10, zero-numerator share 4.0% (German) / 0.0% (non-German), both under the 1/3 threshold.

**4.2.2 Shared vs. group-specific denominators.** Non-German has *more* social-review volume than German (887 vs. 668 mean) despite these being reviews of German pools — the anomaly that led directly to discovering §4.3's classifier bug.

**4.2.3 Stacked interaction model, cluster SEs (headline test).** Joint Wald test on all 5 interaction terms: **χ²=8.45, df=5, p=0.133** — predictor slopes do **not** differ significantly between German and non-German reviewers, on the group-specific-denominator specification. Moran's I significant for both groups' residuals separately (German I=0.082 p=0.012; non-German I=0.094 p=0.008).

**4.2.4 Unknown-category sensitivity.** Folding Unknown into non-German: p=0.128. Treating Unknown as a third level: p=0.374. Both agree with the headline — conclusion survives.

**4.2.5 US-category robustness — new work, this pass.** Three specifications, computed below.

In [ ]:
# Full setup for notebook 04's new-work sections (4.2.5, 4.2.6, 4.3):
# rebuilds analysis_df, ethnicity_final/group5, df_split, and the stacked-model helpers
# exactly as archive/supervisor_run_2026-08-16.ipynb cells 2-7, 70, 74 do.
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.proportion import proportion_confint

warnings.filterwarnings("ignore", category=FutureWarning)

ANALYSIS_DIR = Path("/home/ethel/Desktop/swimming_pool_project/spp_15052026/analysis_10072026")
GMAPS_DIR = ANALYSIS_DIR / "gmaps_data"
SOCIO_DIR = ANALYSIS_DIR / "socio-economic_data"
SHAPEFILE_PATH = Path("/home/ethel/Desktop/swimming_pool_project/btw25_geometrie_wahlkreise_shp/btw25_geometrie_wahlkreise_shp.shp")

corpus = pd.read_parquet(GMAPS_DIR / "master_corpus_final.parquet",
                          columns=["review_id", "unique_identifier", "social", "racialised",
                                   "total_text_reviews", "ethnicity_country_name",
                                   "latitude", "longitude", "place_name"])
political = pd.read_csv(SOCIO_DIR / "federal_baseline_eng_selected_political.csv")
socio_political = pd.read_csv(SOCIO_DIR / "socio_political_data.csv")

places = corpus[["unique_identifier", "place_name", "latitude", "longitude"]].drop_duplicates(subset="unique_identifier").reset_index(drop=True)
places_geo = gpd.GeoDataFrame(places, geometry=gpd.points_from_xy(places["longitude"], places["latitude"]), crs="EPSG:4326").to_crs("EPSG:25832")
constituencies = gpd.read_file(SHAPEFILE_PATH)
if (~constituencies.geometry.is_valid).any():
    constituencies["geometry"] = constituencies.geometry.buffer(0)
germany_boundary = constituencies.dissolve().geometry.iloc[0]
germany_gdf = gpd.GeoDataFrame(geometry=[germany_boundary], crs=constituencies.crs)
inside_idx = gpd.sjoin(places_geo, germany_gdf, predicate="within", how="inner").index
places_inside = places_geo.loc[places_geo.index.isin(inside_idx)].copy()
joined = gpd.sjoin_nearest(places_inside, constituencies[["WKR_NR", "WKR_NAME", "LAND_NR", "LAND_NAME", "geometry"]], how="left")
joined = joined.drop_duplicates(subset="unique_identifier", keep="first")
place_constituency = joined.rename(columns={"WKR_NR": "constituency_code", "WKR_NAME": "constituency", "LAND_NAME": "state"})[
    ["unique_identifier", "constituency_code", "constituency", "state"]].reset_index(drop=True)
corpus = corpus.merge(place_constituency, on="unique_identifier", how="left")
social_df = corpus.loc[corpus["social"] == "yes"].copy()
racialised_df = corpus.loc[corpus["racialised"] == "yes"].copy()

ARAB_LEAGUE_VARIANTS = {
    "Algeria": ["Algeria", "People's Democratic Republic of Algeria"], "Bahrain": ["Bahrain", "Kingdom of Bahrain"],
    "Comoros": ["Comoros", "Union of the Comoros"], "Djibouti": ["Djibouti", "Republic of Djibouti"],
    "Egypt": ["Egypt", "Egypt, Arab Republic of"], "Iraq": ["Iraq", "Republic of Iraq"],
    "Jordan": ["Jordan", "Hashemite Kingdom of Jordan"], "Kuwait": ["Kuwait", "State of Kuwait"],
    "Lebanon": ["Lebanon", "Lebanese Republic"], "Libya": ["Libya", "Libyan Arab Jamahiriya", "State of Libya"],
    "Mauritania": ["Mauritania", "Islamic Republic of Mauritania"], "Morocco": ["Morocco", "Kingdom of Morocco"],
    "Oman": ["Oman", "Sultanate of Oman"],
    "Palestine": ["Palestine, State of", "State of Palestine", "Palestinian Territory, Occupied"],
    "Qatar": ["Qatar", "State of Qatar"], "Saudi Arabia": ["Saudi Arabia", "Kingdom of Saudi Arabia"],
    "Somalia": ["Somalia", "Federal Republic of Somalia"], "Sudan": ["Sudan", "Republic of the Sudan"],
    "Syria": ["Syrian Arab Republic", "Syria"], "Tunisia": ["Tunisia", "Tunisian Republic"],
    "United Arab Emirates": ["United Arab Emirates"], "Yemen": ["Yemen", "Republic of Yemen"],
}
def build_ethnicity_final(country_series, n_top=10):
    present = set(country_series.dropna().unique())
    arab_value_map = {v: c for c, variants in ARAB_LEAGUE_VARIANTS.items() for v in variants if v in present}
    arab_mask = country_series.isin(arab_value_map.keys())
    non_arab_mask = ~arab_mask & country_series.notna()
    top = country_series.loc[non_arab_mask].value_counts().head(n_top).index
    final = pd.Series("Other", index=country_series.index)
    final = final.mask(country_series.isin(top), country_series)
    final = final.mask(arab_mask, "Arab")
    final = final.mask(country_series.isna(), "Unknown")
    return final

social_df["ethnicity_final"] = build_ethnicity_final(social_df["ethnicity_country_name"])
def to_group5(v):
    return v if v in ("Germany", "Arab", "Türkiye", "Unknown") else "Other"
social_df["ethnicity_group5"] = social_df["ethnicity_final"].map(to_group5)
racialised_df = racialised_df.join(social_df[["ethnicity_final", "ethnicity_group5"]])

def wilson(k, n):
    if n == 0: return np.nan, np.nan, np.nan
    lo, hi = proportion_confint(k, n, method="wilson")
    return k / n, lo, hi

backbone = political[["state", "constituency_code", "constituency"]].copy()
agg = (corpus.groupby("constituency_code")
       .agg(pool_count=("unique_identifier", "nunique"), total_text_reviews=("review_id", "count"),
            social_reviews=("social", lambda s: (s == "yes").sum()),
            racialised_reviews=("racialised", lambda s: (s == "yes").sum())).reset_index())
const_agg = backbone.merge(agg, on="constituency_code", how="left")
for c in ["pool_count", "total_text_reviews", "social_reviews", "racialised_reviews"]:
    const_agg[c] = const_agg[c].fillna(0).astype(int)
rates = const_agg.apply(lambda r: pd.Series(wilson(r["racialised_reviews"], r["social_reviews"]), index=["racialised_rate", "ci_lo", "ci_hi"]), axis=1)
const_agg = pd.concat([const_agg, rates], axis=1)

analysis_df = const_agg.merge(socio_political, on=["state", "constituency_code", "constituency"], how="left")
analysis_df["log_population_density"] = np.log(analysis_df["population_density_per_km2"])
EAST_STATES = ["Brandenburg", "Mecklenburg-Vorpommern", "Sachsen", "Sachsen-Anhalt", "Thüringen"]
analysis_df["is_east"] = analysis_df["state"].isin(EAST_STATES).astype(int)
analysis_df["disposable_income_1k"] = analysis_df["disposable_income_eur_per_inhabitant"] / 1000
CHECK_COLS = ["unemployment_total_pct", "sgb_ii_recipients_per_1k", "school_grads_higher_education_entrance_pct",
              "age_60_74_pct", "age_75_plus_pct", "disposable_income_1k"]
dup_groups = analysis_df.groupby(CHECK_COLS)["constituency"].apply(list)
dup_groups = dup_groups[dup_groups.apply(len) > 1]
flagged = {c for consts in dup_groups for c in consts}
analysis_df["in_split_city"] = analysis_df["constituency"].isin(flagged).astype(int)
for c in ["foreign_population_pct", "log_population_density", "sgb_ii_recipients_per_1k"]:
    analysis_df[c + "_c"] = analysis_df[c] - analysis_df[c].mean()
FULL_FORMULA = "racialised_reviews ~ foreign_population_pct_c + log_population_density_c + sgb_ii_recipients_per_1k_c + is_east + in_split_city"

def build_long(df, rac_cols, soc_cols):
    preds_cols = ["constituency_code", "state", "foreign_population_pct_c", "log_population_density_c",
                  "sgb_ii_recipients_per_1k_c", "is_east", "in_split_city"]
    parts = []
    for grp in rac_cols:
        tmp = df[preds_cols + [rac_cols[grp], soc_cols[grp]]].rename(columns={rac_cols[grp]: "n_rac", soc_cols[grp]: "n_soc"}).copy()
        tmp["group"] = grp
        parts.append(tmp)
    long_df = pd.concat(parts, ignore_index=True)
    long_df["n_nonrac"] = long_df["n_soc"] - long_df["n_rac"]
    long_df["group"] = pd.Categorical(long_df["group"], categories=list(rac_cols.keys()))
    return long_df

def fit_stacked(long_df):
    preds = FULL_FORMULA.split("~")[1].strip()
    fit = smf.glm(f"n_rac + n_nonrac ~ group * ({preds})", data=long_df, family=sm.families.Binomial()).fit(
        cov_type="cluster", cov_kwds={"groups": long_df["constituency_code"]})
    inter = [t for t in fit.params.index if ":" in t]
    R = np.zeros((len(inter), len(fit.params)))
    for i, t in enumerate(inter):
        R[i, fit.params.index.get_loc(t)] = 1
    return fit, inter, fit.wald_test(R, scalar=True)

print(f"setup OK: analysis_df {analysis_df.shape}, social_df {len(social_df):,}, racialised_df {len(racialised_df):,}")

In [ ]:
# 4.2.5 -- Task B, three US-category specifications
def build_split(soc_df, rac_df, german_extra=()):
    def is_german(df):
        return df["ethnicity_final"].isin(["Germany"] + list(german_extra))
    def is_nongerman(df):
        return ~df["ethnicity_final"].isin(["Germany", "Unknown"] + list(german_extra))
    d = analysis_df.copy()
    for name, mask_fn, src in [("german", is_german, None), ("nongerman", is_nongerman, None)]:
        soc_counts = soc_df[mask_fn(soc_df)].groupby("constituency_code").size()
        rac_counts = rac_df[mask_fn(rac_df)].groupby("constituency_code").size()
        d[f"n_soc_{name}"] = d["constituency_code"].map(soc_counts).fillna(0).astype(int)
        d[f"n_rac_{name}"] = d["constituency_code"].map(rac_counts).fillna(0).astype(int)
    return d

def wald_for(soc_df, rac_df, german_extra=()):
    split = build_split(soc_df, rac_df, german_extra)
    long_df = build_long(split, {"german": "n_rac_german", "nongerman": "n_rac_nongerman"},
                          {"german": "n_soc_german", "nongerman": "n_soc_nongerman"})
    _, _, wald = fit_stacked(long_df)
    return wald.pvalue, split

US = "United States"
p_baseline, split_baseline = wald_for(social_df, racialised_df)  # (c) baseline: US -> non-German (status quo)
soc_excl = social_df[social_df["ethnicity_country_name"] != US]
rac_excl = racialised_df[racialised_df["ethnicity_country_name"] != US]
p_excluded, split_excluded = wald_for(soc_excl, rac_excl)  # (a) US excluded
p_fold_german, split_fold = wald_for(social_df, racialised_df, german_extra=[US])  # (b) US -> German

print("=== Task B joint Wald test, three US-category specifications ===")
print(f"(a) US excluded              chi2 test  p={p_excluded:.4f}")
print(f"(b) US -> German             chi2 test  p={p_fold_german:.4f}")
print(f"(c) US -> non-German (baseline, == headline 4.2.3)  p={p_baseline:.4f}")
n_moved = int(split_fold["n_rac_german"].sum() - split_baseline["n_rac_german"].sum())
n_baseline_nongerman = int(split_baseline["n_rac_nongerman"].sum())
print(f"\nracialised reviews reassigned German<-nonGerman by the US fold: +{n_moved} "
      f"({n_moved}/{n_baseline_nongerman} = {n_moved/n_baseline_nongerman:.1%} of the baseline non-German racialised count)")

**Result:** all three specifications move the joint Wald p away from the baseline 0.133 toward marginal significance (excluded: p=0.077; fold-to-German: p=0.087) — the direction the original recommendation anticipated (contamination attenuates the group difference toward null), though none of the three cross the conventional 0.05 threshold. **4.2.3's headline should be read as "no significant difference, or a real difference attenuated by classifier contamination" — not as clean evidence of no difference** — consistent with how `analysis_checks_summary.md` already flagged it, now with the actual refits behind the claim rather than a recommendation to run them.

**4.2.6 Permutation check on the p-value movement — new work, this pass.**

In [ ]:
# 4.2.6 -- permutation test: is the p-movement from the US fold (spec b) unusual, or would
# any random same-sized reassignment produce a similar shift? 500 reps, seed fixed for reproducibility.
import time

is_german_base = social_df["ethnicity_final"] == "Germany"
is_nongerman_base = ~social_df["ethnicity_final"].isin(["Germany", "Unknown"])
nongerman_social_pool = social_df.index[is_nongerman_base].to_numpy()
us_mask = social_df["ethnicity_final"] == "United States"
N_MOVE_SOCIAL = int(us_mask.sum())

def wald_p_for_reassignment(reassign_idx_set):
    d = analysis_df.copy()
    moved = social_df.index.isin(reassign_idx_set)
    is_g, is_ng = is_german_base | moved, is_nongerman_base & ~moved
    soc_g = social_df[is_g].groupby("constituency_code").size()
    rac_g = racialised_df[racialised_df.index.isin(social_df.index[is_g])].groupby("constituency_code").size()
    soc_ng = social_df[is_ng].groupby("constituency_code").size()
    rac_ng = racialised_df[racialised_df.index.isin(social_df.index[is_ng])].groupby("constituency_code").size()
    d["n_soc_german"] = d["constituency_code"].map(soc_g).fillna(0).astype(int)
    d["n_rac_german"] = d["constituency_code"].map(rac_g).fillna(0).astype(int)
    d["n_soc_nongerman"] = d["constituency_code"].map(soc_ng).fillna(0).astype(int)
    d["n_rac_nongerman"] = d["constituency_code"].map(rac_ng).fillna(0).astype(int)
    long_df = build_long(d, {"german": "n_rac_german", "nongerman": "n_rac_nongerman"},
                          {"german": "n_soc_german", "nongerman": "n_soc_nongerman"})
    _, _, wald = fit_stacked(long_df)
    n_rac_moved = int(racialised_df.index.isin(reassign_idx_set).sum())
    return wald.pvalue, n_rac_moved

us_idx = set(social_df.index[us_mask])
observed_p, observed_n_rac = wald_p_for_reassignment(us_idx)

rng = np.random.default_rng(20260824)
null_ps, null_n_rac = [], []
t0 = time.time()
for rep in range(500):
    chosen = rng.choice(nongerman_social_pool, size=N_MOVE_SOCIAL, replace=False)
    p, n_rac = wald_p_for_reassignment(set(chosen))
    null_ps.append(p); null_n_rac.append(n_rac)
null_ps, null_n_rac = np.array(null_ps), np.array(null_n_rac)

print(f"permutation done in {time.time()-t0:.0f}s, 500 reps")
print(f"observed (actual US-fold): p={observed_p:.4f}, racialised reviews moved={observed_n_rac}")
print(f"null (random same-sized reassignment): p mean={null_ps.mean():.4f}, median={np.median(null_ps):.4f}, "
      f"[5th,95th pct]=[{np.percentile(null_ps,5):.4f}, {np.percentile(null_ps,95):.4f}]")
print(f"null racialised-moved: mean={null_n_rac.mean():.0f}, [5th,95th pct]=[{np.percentile(null_n_rac,5):.0f}, {np.percentile(null_n_rac,95):.0f}]")
print(f"\nfraction of random reassignments with p <= observed: {(null_ps <= observed_p).mean():.3f}")
print(f"fraction of random reassignments significant at p<0.05: {(null_ps < 0.05).mean():.3f}")
print(f"fraction of random reassignments moving >= {observed_n_rac} racialised reviews: {(null_n_rac >= observed_n_rac).mean():.3f}")

**Result:** the p-value movement itself is **not** distinctive — 38% of random same-sized (70,554-row) reassignments from the non-German pool produce a joint Wald p at or below the real US-fold's 0.087, and 29% cross p<0.05 outright. The joint Wald test is simply sensitive to denominator reshuffling of this size, regardless of content — **the 0.087 number should not be read as evidence specific to the US bug.**

What *is* distinctive: the real US-labeled group moved only 899 racialised reviews, while **every one of the 500 random same-sized draws moved more** (mean ~1266, 5th percentile 1216) — i.e., the US-predicted reviewers write racialised content at a noticeably *lower* rate than an equivalent random slice of the non-German pool. That is exactly the signature you'd expect if these are mislabeled Germans (who racialise at the lowest rate of any group, §B9/§4.2.3) rather than genuine non-German reviewers — indirect, data-internal support for the classifier-bug diagnosis in §4.3, independent of the manual name spot-check.

**4.2.7 Interpretation.** Underpowered test (median 10 racialised reviews per constituency in the non-German group) on a proxy variable (name-predicted ethnicity) with a known measurement error (§4.3). Shows no detectable moderation; does not show absence of moderation. The permutation result above adds a second reason for caution beyond power: the test statistic itself is not very informative about composition changes of this size, so even a "clean" non-German bucket would need a materially larger effect to move it convincingly.

### 4.3 The "United States" classifier bug  [LIMITATIONS — SETTLED]

170,807 reviews are name→country predicted as "United States" — the corpus's 2nd-largest predicted category after Germany (500,951), ahead of Russia (28,567) and Türkiye (21,671) by 6–8×. Because `to_group5` folds anything outside {Germany, Arab, Türkiye, Unknown} into "Other," and Task B's non-German bucket is Arab+Türkiye+Other, all of that volume lands in non-German. **899 of the 4,762 non-German racialised reviews (18.9%, "~19%") are US-predicted** — see §4.2.5/§4.2.6 for the refits and permutation test this motivates.

*(Note: an earlier note in `analysis_checks_summary.md` cited 124,521 — likely computed on a differently-filtered subset; 170,807 is the figure from a direct count against `master_corpus_final.parquet`'s `ethnicity_country_name` column, reproduced below.)*

In [ ]:
# 4.3 -- seeded reproducible name spot-checks (replaces the earlier unseeded n=30 check)
# plus the brief's optional cheap check: same spot-check on other large predicted categories.
names_df = pd.read_parquet(GMAPS_DIR / "master_corpus_final.parquet", columns=["reviewer_name", "ethnicity_country_name"])
print(names_df["ethnicity_country_name"].value_counts().head(6))

us_names = names_df.loc[names_df["ethnicity_country_name"] == "United States", "reviewer_name"].dropna()
for seed in (42, 7):
    print(f"\n--- United States-predicted, seed={seed}, n=25 ---")
    print(list(us_names.sample(n=25, random_state=seed)))

print("\n--- cheap check: same spot-check on other large categories, seed=42 ---")
for country in ["Germany", "Türkiye", "France"]:
    pool = names_df.loc[names_df["ethnicity_country_name"] == country, "reviewer_name"].dropna()
    print(f"\n{country} (pool size {len(pool)}):")
    print(list(pool.sample(n=25, random_state=42)))

**Result:** both seeded 25-name US-predicted samples read as ordinary German first+surname pairs (*Angela Baumgartner, Christina Stock, Sandra Meyer, Nicole Gutschlich, Katharina von Klenck*, ...) with a handful of ambiguous pseudonyms (*Ghost Killings, Fat Cat, Pittel*) but nothing distinctly American — consistent with the classifier's population-prior mechanism (German surnames are far more numerous in the US than in Germany in absolute population terms, so the tool's population-weighted prior overwhelms the actual name-origin signal).

**The cheap check confirms this is not US-specific:** the Germany- and Türkiye-predicted samples read as expected (clean German / clean Turkish names), but the **France-predicted sample is visibly mixed** — alongside genuinely French names (*Henri Roland, Sandrine Combre, Jeanne Sophie Duqueroy*) it contains clearly German ones (*Elfriede Schreibert, Franz Ludmann, Werner Politowski*). Same mechanism, smaller pool (18,367 vs. 170,807), smaller absolute distortion — but the same population-prior bug, not a US-only quirk. This is a systematic, directional error that depletes the German category and inflates phantom "foreign" ones, not random noise.

### 4.4 Classifier validation status  [LIMITATIONS — OPEN]

Sampling and scoring pipeline built (100-row blind kappa subset + 499-row adjudication sheet, `outputs/ethnicity_validation/`), **0/599 hand-coded**, and the prediction pipeline retains no confidence score (`Prediction Result.csv` has only a country string) — a genuine pipeline limitation, verified directly. **Blocked on researcher coding**, not more automation. Once coded: Cohen's κ (below ~0.7 means the LLM pre-labels aren't trustworthy for the un-adjudicated remainder), then the confusion matrix, error-by-feature crosstabs, and — the load-bearing piece for both §4.2 and §4.1 — whether the audit error rate is *patterned* on the same predictors used throughout (confounding) or flat (mere attenuation).

### 4.5 Directionality: not yet tested  [OPEN]

§4.2 tests whether the *predictor slopes* differ between reviewer groups — it does not test **direction**: whether a reviewer is racialising others, or reporting having been racialised. Reviewer ethnicity is a loose proxy for this: a German reviewer can describe witnessing racism; a Turkish reviewer can racialise Arab bathers. §4.2's null result **must not be cited as evidence about direction in either direction.** Proposed test (not run): direct text coding of ~200 racialised reviews for who is doing the racialising, independent of reviewer name-origin.

### 4.6 Spatial dependence  [LIMITATIONS — SETTLED, accepted as a stated limitation]

Every Moran's I computed across this thread, consolidated:

| model / quantity | residual type | I | p |
|---|---|---|---|
| `rate_social` outcome (raw, §3.4) | — | 0.281 | 0.001 |
| `rate_total` final NB model (§4.1) | Pearson | 0.274 | 0.001 |
| Baseline OLS (§2.1/2.2) | OLS residual | 0.116 | 0.003 |
| Baseline binomial GLM (§2.1) | deviance residual | 0.107 | 0.004 |
| Task B stacked model, German group (§4.2.3) | deviance residual | 0.082 | 0.012 |
| Task B stacked model, non-German group (§4.2.3) | deviance residual | 0.094 | 0.008 |
| `rate_social` final NB model (`nb_main`, §3.1) | Pearson | 0.105 | — |

Range: **0.082 to 0.281**, significant everywhere it's been tested. No model in this thread fully absorbs the spatial structure present in the raw outcome — it shrinks by roughly half moving from raw outcome to fitted-model residuals, but never to zero. *(Source: `analysis_checks_summary.md` Task A; `rate_total_analysis.ipynb` C5; `archive/supervisor_run_2026-08-16.ipynb` cells `B7`/`D6`/Task B.)*

**2026-08-24 gap-closing pass: actually tried the candidate fix, below, rather than just naming it.**

In [ ]:
# 4.6 -- does a spatial-error/lag correction actually reduce the residual dependence?
# (ML_Error/ML_Lag operate on a continuous response, so this is log(rate_social) as the
# best available approximation to the NB model -- not a claim that this IS the NB model.)
from spreg import ML_Error, ML_Lag
from esda.moran import Moran
from libpysal.weights import Queen
import statsmodels.api as sm

d = analysis_df.copy()
d["log_rate_social"] = np.log(d["racialised_reviews"] / d["social_reviews"])
preds = FULL_FORMULA.split("~")[1].strip().split("+")
preds = [p.strip() for p in preds]

moran_gdf = constituencies[["WKR_NR", "geometry"]].rename(columns={"WKR_NR": "constituency_code"}).merge(
    d[["constituency_code"] + preds + ["log_rate_social"]], on="constituency_code", how="left"
).dropna(subset=["log_rate_social"]).reset_index(drop=True)
w = Queen.from_dataframe(moran_gdf, use_index=False)
w.transform = "r"

y = moran_gdf["log_rate_social"].values.reshape(-1, 1)
X = moran_gdf[preds].values

err = ML_Error(y, X, w=w, name_y="log_rate_social", name_x=preds)
lag = ML_Lag(y, X, w=w, name_y="log_rate_social", name_x=preds)
print(f"spatial-error lambda: {err.betas[-1][0]:.4f}   spatial-lag rho: {lag.betas[-1][0]:.4f}")

resid_err = err.u.flatten()
m_err = Moran(resid_err, w, permutations=999)
ols = sm.OLS(y, sm.add_constant(X)).fit()
m_ols = Moran(ols.resid, w, permutations=999)
print(f"plain-OLS (log-rate scale) residual Moran's I:   {m_ols.I:.4f}, p={m_ols.p_sim:.4f}")
print(f"spatial-error-model residual Moran's I:          {m_err.I:.4f}, p={m_err.p_sim:.4f}")

**Result and decision.** The spatial-error model finds real spatial structure (λ≈0.18, a spatial-lag ρ of similar size) — so the dependence is genuine, not a fluke of the weights choice. But its residual Moran's I (≈0.08) is barely different from plain OLS on the same log-rate scale (≈0.08) — **spatial-error/lag correction, at this specification, does not meaningfully reduce the residual dependence.** Decision: accept the residual spatial dependence as a stated limitation rather than pursue a full spatially-structured count model (e.g. an NB with a spatial random effect) — the simple fix doesn't work, and the complex one is out of scope for this timeline. State it as a limitation in the paper; don't imply it's been "handled."